In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_parquet("unlp_sharedtask_dataset/train.parquet")
df.shape

(3822, 6)

In [5]:
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [8]:
df.lang.value_counts()

lang
uk    2147
ru    1675
Name: count, dtype: int64

In [7]:
pd.crosstab(df.lang, df.manipulative)

manipulative,False,True
lang,,
ru,421,1254
uk,812,1335


In [ ]:
train_ratio = 0.8
eval_ratio = 0.1
test_ratio = 0.1

test_size_num = int(test_ratio * len(df))

uk_df = df[df['lang'] == 'uk']

if len(uk_df) < test_size_num:
    raise ValueError("Not enough rows with lang=='uk' to form the test set.")

test_df, _ = train_test_split(
    uk_df,
    train_size=test_size_num,
    stratify=uk_df['manipulative'],
    random_state=42
)

remaining_df = df.drop(test_df.index)

train_df, eval_df = train_test_split(
    remaining_df,
    test_size=eval_ratio/(train_ratio + eval_ratio),
    stratify=remaining_df['manipulative'],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Eval shape:", eval_df.shape)

train_df = train_df[['content', 'manipulative']].rename(columns={'content': 'text', 'manipulative': 'label'})
test_df  = test_df[['content', 'manipulative']].rename(columns={'content': 'text', 'manipulative': 'label'})
eval_df  = eval_df[['content', 'manipulative']].rename(columns={'content': 'text', 'manipulative': 'label'})

# Save the resulting splits to CSV files
train_df.to_csv('unlp_sharedtask_dataset/train.csv', index=False)
test_df.to_csv('unlp_sharedtask_dataset/test.csv', index=False)
eval_df.to_csv('unlp_sharedtask_dataset/eval.csv', index=False)


Train shape: (3057, 6)
Test shape: (382, 6)
Eval shape: (383, 6)
